In [ ]:
"""
postprocess_gfm_flood.ipynb
===========================
Post-processing of GFM monthly flood-day GeoTIFFs.

Reads the per-month flood-day rasters produced by `gfm_flood_days.ipynb`
and computes three population-flood indicators per admin area:

  1. Impacted population rate — % of admin population living under the
     any-day-flooded mask over the selected 12-month window.
  2. Pop-weighted average flood days (total-population denominator) —
     average flood-day burden per person in the admin area.
  3. Pop-weighted average flood days (intersection denominator) —
     average flood-day burden among the subset of population
     that experienced any flooding.

Outputs
-------
  - Compressed GeoJSON of the any-day-flooded mask (country-wide)
  - One Excel file per indicator + binarization columns
  - One choropleth PNG (one per indicator)
  - A JSON run log with processing parameters and per-admin QC statistics

Technical approach
------------------
  - Monthly GeoTIFFs are stacked and summed into a cumulative flood-day
    raster, with per-month reprojection to a common reference grid
    (handles the ±1-2 pixel grid drift, if any, during the extraction step).
  - WorldPop constrained population is reprojected from EPSG:4326 onto
    the flood grid (UTM CRS, 100m) using nearest-neighbour resampling
    to preserve population counts.
  - Admin boundaries are rasterized onto the same grid, enabling fast
    numpy/pandas zonal aggregation without intermediate vector operations.
  - The vectorised flood mask (GeoJSON) is a byproduct for visualisation
    and downstream use; it is NOT used as the computational backbone.

Future refactoring note
-----------------------
  The binarization, choropleth, and logging functions are written to be
  liftable into a shared `wia_utils` module for reuse across ACLED,
  exposure, RWI, and other hazard post-processing workflows.
"""

In [ ]:
# 0. IMPORTS

import json
import logging
from datetime import date, datetime, timezone, timedelta
from pathlib import Path

from dateutil.relativedelta import relativedelta
# from geojson_rewind import rewind
import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pycountry
import rasterio
from rasterio.features import rasterize#, shapes
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds, xy
# from shapely import make_valid
# from shapely.geometry import shape as shapely_shape
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union
import subprocess
import tempfile
# import zipfile


In [ ]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


##### 1. CONFIGURATION — edit this section for your run

In [ ]:
# ── Folder structure ──
base_dir = "./"
data_in_folder = base_dir + "data_in/"
data_in_shapes = base_dir + "data_in/shapes/"

# GFM rasters produced by gfm_flood_days.ipynb
gfm_raster_dir = Path(data_in_folder + "Raster_Datasets/EU_GFM")

# WorldPop constrained population rasters
wpop_dir = data_in_folder + "WorldPop_rasters/G2_CN_POP_R25A_100m"

# Output folders
maps_hazards_folder = base_dir + "data_out/maps/hazards/"
output_log_dir = base_dir + "data_out/logs/"
Path(maps_hazards_folder).mkdir(parents=True, exist_ok=True)
Path(output_log_dir).mkdir(parents=True, exist_ok=True)


In [ ]:
# ── Country / admin / time selection ──
# NOTE: selected countries for WIA execution
wia_countries_exercise = [
    # "Kenya",
    # "Mali",
    # "Benin",
    # "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    # "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    # "Congo, The Democratic Republic of the",
    # "Haiti",
    # "Somalia",
    "Sudan",
    # "Yemen",
    # "Saint Vincent and the Grenadines",
    # "Grenada",
]
country = wia_countries_exercise[0]

c_iso3 = (
    pycountry.countries.search_fuzzy(country)[0].alpha_3
    if country != "Niger"
    # introduced Niger exception (known case to avoid Nigeria)
    else pycountry.countries.search_fuzzy(country)[1].alpha_3
)

# NOTE: select admin level and population year
admin_level = "2"
year = f"{date.today().year - 1}"

# Column names (assume harmonized across CODs)
pcode_col = f"adm{admin_level}_pcode"
name_col = f"adm{admin_level}_name"

# ── Flood nodata value (from GFM extraction script) ──
FLOOD_NODATA = 255


In [ ]:
# ── Binarization / map settings ──
# Impact (indicator 1): higher % impacted = worse → always "above"
# Weighted averages (indicators 2 & 3): higher flood days = worse → "above"
NODATA_GREY = "Gray"
CSCALE_FLOOD = "YlOrRd"  # higher = worse = red

# Flood Map resolution in degrees (~500m at equator ≈ 0.0045°)
# This 5/10x downsampling from 100m keeps the visual quality
# at country-map zoom while reducing point count by ~25/100x
VIZ_RES_DEG = 0.009 # 0.0045 #


In [ ]:
# -- Time window for consultation --
# NOTE: select last year or last 12 months of data
time_filter = "last_year" # "last_12_months" #

# Uses last updated Friday like Acled Updates (assumes fresh data on Wednesdays)
today = date.today()
weekday = today.weekday()
last_updated_friday = today - timedelta(
    days=(weekday - 4) % 7 + (0 if weekday in [2, 3] else 7)
)

# end of previous month with completed updates or end of last year
last_updated_month = (
    last_updated_friday - timedelta(days=last_updated_friday.day)
    if time_filter == "last_12_months"
    else date(today.year - 1, 12, 31)
)
# 12 last completed months with updates
start_query_date = last_updated_month - relativedelta(months=12) + timedelta(days=1)


##### 2. LOAD ADMIN BOUNDARIES

In [ ]:
# geojson file name
geo_filename = f"geojson_{c_iso3}_adm{admin_level}"
geojson_file = [
    f.name
    for f in Path(data_in_shapes).iterdir()
    if f.name.endswith(".zip") and geo_filename in f.name
]

# read country shape geojson into gdf
gdf = gpd.read_file(
    data_in_shapes + geojson_file[0]
).to_crs("EPSG:4326")

geojson_dict = json.loads(gdf.to_json())

log.info(
    f"Loaded {len(gdf)} admin{admin_level} features for {c_iso3} "
    f"from {geojson_file[0]}"
)


##### 3. DISCOVER AND STACK FLOOD GeoTIFFs

In [ ]:
# Iterate calendar months in the time window datetime range
# NOTE: could be lifted into shared utils for reuse across WIA hazards workflows
def month_range(start: date, end: date):
    """Yield (month_start, month_end) tuples for each calendar month."""
    cursor = start
    while cursor < end:
        next_month = cursor + relativedelta(months=1)
        # month_end is last day of month
        yield cursor, next_month - timedelta(days=1)
        cursor = next_month


In [ ]:
# Discover GeoTIFFs matching country ISO3
flood_tifs = sorted(gfm_raster_dir.glob(f"flood_days_{c_iso3}_*.tif"))

# get the 12-month window monthly splits
monthly_tuples = month_range(start_query_date, last_updated_month)

# list of GeoTIFFs in the expected 12-month window
flood_tifs_in_time = sorted([
    t
    for m_start, _ in monthly_tuples
    for t in flood_tifs
    if m_start.strftime("%Y-%m") in t.name
])

if len(flood_tifs) < 12:
    # verify total number of monthly GeoTIFFs
    raise ValueError(
        f"Expected 12 monthly flood-day GeoTIFFs for {c_iso3} but found {len(flood_tifs)}."
    )
elif len(flood_tifs_in_time) < 12:
    # verify that the GeoTIFFs cover the expected 12-month window
    raise ValueError(
        f"Expected 12 monthly flood-day GeoTIFFs for {c_iso3} but found {len(flood_tifs_in_time)}."
    )

log.info(
    f"Found {len(flood_tifs)} monthly flood-day GeoTIFFs for {c_iso3}.\n"
    f"Processing the following files in the selected time window '{time_filter}':"
)
for t in flood_tifs_in_time:
    log.info(f"  {t.name}")


In [ ]:
# Use the first GeoTIFF as the reference grid
# All other months will be reprojected to this grid if needed
with rasterio.open(flood_tifs_in_time[0]) as ref_src:
    ref_profile = ref_src.profile.copy()
    ref_crs = ref_src.crs
    ref_transform = ref_src.transform
    ref_shape = (ref_src.height, ref_src.width)
    ref_bounds = ref_src.bounds

log.info(f"Reference grid: {ref_shape[1]}x{ref_shape[0]} px, CRS={ref_crs}")
log.info(f"Reference bounds: {ref_bounds}")


In [ ]:
# Stack all months: sum flood days into a cumulative raster
# Handles grid drift by reprojecting each month to the reference grid
cumulative_flood_days = np.zeros(ref_shape, dtype=np.float32)
months_loaded = []

for tif_path in flood_tifs_in_time:
    with rasterio.open(tif_path) as src:
        src_data = src.read(1)
        src_shape = (src.height, src.width)
        src_transform = src.transform
        src_nodata = src.nodata if src.nodata is not None else FLOOD_NODATA

        # Check for grid drift: if shape or transform differ, reproject
        needs_reproject = (
            src_shape != ref_shape or src_transform != ref_transform
        )

        if needs_reproject:
            log.warning(
                f"  Grid drift in {tif_path.name}: "
                f"{src_shape[1]}x{src_shape[0]} vs ref {ref_shape[1]}x{ref_shape[0]} — reprojecting"
            )
            aligned = np.full(ref_shape, FLOOD_NODATA, dtype=np.uint8)
            reproject(
                source=src_data,
                destination=aligned,
                src_transform=src_transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                resampling=Resampling.nearest,
                src_nodata=src_nodata,
                dst_nodata=FLOOD_NODATA,
            )
            data = aligned
        else:
            data = src_data

    # Accumulate: treat nodata as 0 flood days
    valid = data != FLOOD_NODATA
    cumulative_flood_days[valid] += data[valid].astype(np.float32)

    # Extract month label from filename
    # Expected pattern: flood_days_{ISO3}_{YYYY-MM}_EPSG_{code}.tif
    month_label = tif_path.stem.split("_")[3]  # YYYY-MM
    months_loaded.append(month_label)
    log.info(f"  Stacked {tif_path.name} (reproject={needs_reproject})")

log.info(
    f"Cumulative flood-day raster: "
    f"max={cumulative_flood_days.max():.0f} days, "
    f"{(cumulative_flood_days > 0).sum():,} flooded pixels"
)


##### 4.a CREATE ANY-FLOODED BINARY MASK
NOTE: export GeoJSON commented, raster-based flood visualization

In [ ]:
# Binary mask: any pixel flooded >= 1 day in the whole time frame
any_flooded_mask = (cumulative_flood_days > 0).astype(np.uint8)

n_flooded_pixels = int(any_flooded_mask.sum())
log.info(f"Any-flooded mask: {n_flooded_pixels:,} pixels")


In [ ]:
# NOTE: left commented, final product for visualization would be raster-based
# # Vectorise the mask into polygons
# # rasterio.features.shapes yields (geometry_dict, value) tuples
# flood_polygons = []
# for geom_dict, value in shapes(
#     any_flooded_mask,
#     mask=any_flooded_mask == 1,
#     transform=ref_transform,
# ):
#     if value == 1:
#         flood_polygons.append(shapely_shape(geom_dict))

# if flood_polygons:
#     flood_mask_gdf = gpd.GeoDataFrame(
#         {"flood": [1] * len(flood_polygons)},
#         geometry=flood_polygons,
#         crs=ref_crs,
#     )
#     log.info(f"Vectorised flood mask: {len(flood_polygons)} polygon(s)")
# else:
#     flood_mask_gdf = gpd.GeoDataFrame(
#         columns=["flood", "geometry"], crs=ref_crs,
#     )
#     log.warning("No flooded pixels found — empty flood mask.")


In [ ]:
# NOTE: left commented, final product for visualization would be raster-based
# # Export flood mask as gzipped GeoJSON
# # Reproject to EPSG:4326 for interoperability
# mask_4326 = flood_mask_gdf.to_crs("EPSG:4326")
# geojson_str = mask_4326.to_json()
# geojson_bytes = geojson_str.encode("utf-8")

# uncompressed_mb = len(geojson_bytes) / 1e6
# log.info(f"Flood mask GeoJSON uncompressed size: {uncompressed_mb:.1f} MB")

# # include time window in filename
# mask_filename = (
#     f"{c_iso3}_{start_query_date.strftime('%Y-%m')}_{last_updated_month.strftime('%Y-%m')}_GFM_flood_mask.geojson.gz"
# )
# mask_path = Path(maps_hazards_folder) / mask_filename

# with gzip.open(mask_path, "wb") as f:
#     f.write(geojson_bytes)

# compressed_mb = mask_path.stat().st_size / 1e6
# log.info(f"Compressed GeoJSON saved: {mask_path.name} ({compressed_mb:.2f} MB)")

# if uncompressed_mb > 50:
#     log.warning(
#         f"  Uncompressed GeoJSON is {uncompressed_mb:.0f} MB — consider running "
#         f"a simplification algorithm (e.g. mapshaper -simplify) before "
#         f"loading in web viewers or GIS tools."
#     )


##### 4.b Helper Functions: flooded area geojson for visualization
NOTE: Not suitable for complex geometries, consider profiling & refactoring

Qualitative visualization, area impacted by flood: raster-based

In [ ]:
def geo_keep_polygonal(geom):
    """
    Keep only polygonal parts from a geometry.
    """
    if geom is None or geom.is_empty:
        return None

    if isinstance(geom, (Polygon, MultiPolygon)):
        return geom

    if isinstance(geom, GeometryCollection):
        polys = [
            g for g in geom.geoms
            if isinstance(g, (Polygon, MultiPolygon)) and not g.is_empty
        ]
        if not polys:
            return None
        return unary_union(polys)

    return None


In [ ]:
def geo_remove_small(geom, min_area_m2: float):
    """
    Remove polygon parts smaller than min_area_m2.
    Input geometry must already be in a metric CRS.
    """
    if geom is None or geom.is_empty:
        return geom

    if isinstance(geom, Polygon):
        return geom if geom.area >= min_area_m2 else None

    if isinstance(geom, MultiPolygon):
        kept = [p for p in geom.geoms if p.area >= min_area_m2]
        if not kept:
            return None
        return unary_union(kept)

    return geom


In [ ]:
def mapshaper_simplify(
        gdf: gpd.GeoDataFrame, retain_pct: int
    ) -> gpd.GeoDataFrame:
    """
    Simplify using mapshaper.
    retain_pct is the percentage of points to retain (1-100).
    """
    if not (1 <= retain_pct <= 100):
        raise ValueError("retain_pct must be between 1 and 100.")

    with tempfile.NamedTemporaryFile(suffix=".geojson", delete=False) as tmp_in, \
        tempfile.NamedTemporaryFile(
            suffix=".geojson",
            mode="w",
            encoding='utf-8',
            delete=False,
        ) as tmp_out:

        in_path = Path(tmp_in.name)
        out_path = Path(tmp_out.name)

    try:
        # create temp geojson
        gdf.to_file(in_path, driver="GeoJSON")

        # Use mapshaper to perform clean & topological simplifications
        subprocess.check_call([
            "mapshaper",
            "-i",
            str(in_path),
            "-clean",
            "-simplify",
            f"{retain_pct}%",
            "keep-shapes",
            "-o",
            "format=geojson",
            str(out_path),
        ])

        # read back the simplified file from temp out
        out_gdf = gpd.read_file(out_path)
        if gdf.crs is not None:
            out_gdf = out_gdf.set_crs(gdf.crs, allow_override=True)
        return out_gdf

    finally:
        if in_path.exists():
            in_path.unlink()
        if out_path.exists():
            out_path.unlink()


In [ ]:
def build_flood_display_footprint(
    flood_gdf: gpd.GeoDataFrame,
    min_area_m2: float = 50000,
    amplify_m: float = 0,
    mapshaper_retain_pct: int = 10,
) -> gpd.GeoDataFrame:
    """
    Build a display-friendly flood footprint from gdf flood mask.

    Parameters
    ----------
    flood_gdf : GeoDataFrame
        Input flood polygons.
        NOTE: assumed to be in local metric CRS
    min_area_m2 : float
        Remove polygon parts smaller than this area before visualization.
    amplify_m : float
        Positive buffer in meters to visually enlarge the footprint.
    mapshaper_retain_pct : int
        Percent of points retained.

    Returns
    -------
    GeoDataFrame
        A simplified/augmented display geometry in EPSG:4326.
    """

    if flood_gdf.empty:
        # Assume is the previously created empty flood gdf
        return flood_gdf

    # NOTE: defensive as previous ref_crs not None expected
    if flood_gdf.crs is None:
        raise ValueError("Input GeoDataFrame must have a CRS.")
    
    # Keep only valid polygonal geometry
    # NOTE: some may be just defensive
    work = flood_gdf.copy()
    work = work[~work.geometry.isna()].copy()
    # topology cleaning (alternatively buffer(0) to clean up slivers)
    work["geometry"] = work.geometry.make_valid()
    work["geometry"] = work.geometry.apply(geo_keep_polygonal)
    work = work[work.geometry.notna() & ~work.geometry.is_empty].copy()

    # NOTE: defensive repeated
    if work.empty:
        # return empty flood gdf
        return gpd.GeoDataFrame(
            columns=["flood", "geometry"], crs=ref_crs,
        )

    # Dissolve everything into one display footprint
    merged = unary_union(work.geometry.tolist())
    # NOTE: defensive repeated cleanup
    merged = make_valid(merged)
    merged = geo_keep_polygonal(merged)

    # NOTE: defensive repeated
    if merged is None or merged.is_empty:
        # return empty flood gdf
        return gpd.GeoDataFrame(
            columns=["flood", "geometry"], crs=ref_crs,
        )
    
    # Remove tiny pieces before buffering / simplification
    merged = geo_remove_small(merged, min_area_m2)
    if merged is None or merged.is_empty:
        # return empty flood gdf
        return gpd.GeoDataFrame(
            columns=["flood", "geometry"], crs=ref_crs,
        )
    
    # Optional visual enlargement
    if amplify_m > 0:
        merged = merged.buffer(amplify_m)
        # NOTE: defensive repeated cleanup
        merged = make_valid(merged)
        merged = geo_keep_polygonal(merged)

    # Create the display GeoDataFrame with a single feature
    display_gdf = gpd.GeoDataFrame(
        {"flood": [1]}, geometry=[merged], crs=ref_crs,
    )

    # Simplify the display GeoDataFrame using mapshaper
    display_gdf = mapshaper_simplify(
        display_gdf,
        mapshaper_retain_pct,
    )

    # NOTE: defensive repeated final cleanup
    display_gdf["geometry"] = display_gdf.geometry.make_valid()
    display_gdf["geometry"] = display_gdf.geometry.apply(geo_keep_polygonal)
    display_gdf = display_gdf[
        display_gdf.geometry.notna() & ~display_gdf.geometry.is_empty
    ].copy()

    # NOTE: defensive repeated
    if display_gdf.empty:
        # return empty flood gdf
        return gpd.GeoDataFrame(
            columns=["flood", "geometry"], crs=ref_crs,
        )
    
    # return reprojected to 4326
    return display_gdf.to_crs('EPSG:4326').reset_index(drop=True)


In [ ]:
# NOTE: left commented, final product for visualization would be raster-based
# # display footprint builder with selected parameters
# min_area_rem = 250000   # 5x5 pixels at 100m resolution
# amp_factor = 1000       # 10 times pixel size
# shape_simp = 10         # % of removable points to retain

# flood_display_gdf = build_flood_display_footprint(
#     flood_mask_gdf,
#     min_area_m2=min_area_rem,
#     amplify_m=amp_factor,
#     mapshaper_retain_pct=shape_simp,
# )

# log.info(
#     "Flood mask GeoJSON simplified with parameters:\n"
#     f"  - Minimum area: {min_area_rem} m^2\n"
#     f"  - Amplify factor: {amp_factor} m\n"
#     f"  - Shape simplification: {shape_simp}%\n"
# )


In [ ]:
# NOTE: left commented, final product for visualization would be raster-based
# # ensure display geojson with rewinded shapes
# flood_display_geojson_out = rewind(
#     json.loads(gdf.to_json()),
#     rfc7946=False,
# )

# geo_filename = (
#     f"{c_iso3}_{start_query_date.strftime('%Y-%m')}_"
#     f"{last_updated_month.strftime('%Y-%m')}_"
#     f"{min_area_rem}_m2_"
#     f"{amp_factor}_m_"
#     f"simpl{shape_simp}_"
#     f"visual_flood"
# )
# geo_path = Path(maps_hazards_folder) / (geo_filename + ".geojson.zip")

# # export zip file with geojson
# with zipfile.ZipFile(
#     geo_path,
#     'w',
#     zipfile.ZIP_DEFLATED,
# ) as zip_file:
#     # write geojson to zip file
#     zip_file.writestr(
#         f"{geo_filename}.geojson",
#         json.dumps(flood_display_geojson_out, ensure_ascii=False),
#     )

# compressed_mb = geo_path.stat().st_size / 1e6
# log.info(f"Visualization GeoJSON saved: {geo_path.name} ({compressed_mb:.2f} MB)")


##### 5. REPROJECT WORLDPOP TO FLOOD GRID

In [ ]:
# Identify WorldPop raster for country/year
total_pop_rasters = [
    f.name
    for f in Path(wpop_dir).iterdir()
    if f.is_file()
]

pop_raster_name = next(
    (
        f for f in total_pop_rasters
        if (c_iso3.lower() in f) and (f"_{year}_" in f)
    ),
    None,
)

if pop_raster_name is None:
    raise FileNotFoundError(
        f"No WorldPop raster found for {c_iso3} / {year} in {wpop_dir}"
    )

pop_raster_path = str(Path(wpop_dir) / pop_raster_name)
log.info(f"WorldPop raster: {pop_raster_name}")


In [ ]:
# Reproject WorldPop (EPSG:4326, ~100m) onto the flood grid (UTM, 100m)
# Using nearest-neighbour to preserve population counts (no fractional people)
pop_aligned = np.full(ref_shape, np.nan, dtype=np.float32)

with rasterio.open(pop_raster_path) as pop_src:
    pop_nodata = pop_src.nodata
    log.info(
        f"  Source CRS: {pop_src.crs}, shape: {pop_src.width}x{pop_src.height}, "
        f"nodata: {pop_nodata}"
    )

    reproject(
        source=rasterio.band(pop_src, 1),
        destination=pop_aligned,
        src_transform=pop_src.transform,
        src_crs=pop_src.crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        resampling=Resampling.nearest,
        src_nodata=pop_nodata,
        dst_nodata=np.nan,
    )

# Replace any residual source nodata values with NaN
if pop_nodata is not None:
    pop_aligned[pop_aligned == pop_nodata] = np.nan

# Constrained pop: nodata means no population → treat as zero
# consistent with nodata_means_zero for population
pop_with_data = ~np.isnan(pop_aligned)
pop_zero_filled = np.where(pop_with_data, pop_aligned, 0.0)

log.info(
    f"  Pop reprojected: {pop_with_data.sum():,} data pixels, "
    f"total pop = {np.nansum(pop_aligned):,.0f}"
)


##### 6. RASTERIZE ADMIN BOUNDARIES

In [ ]:
# Project admin boundaries to flood CRS
gdf_proj = gdf.to_crs(ref_crs)

# Create integer ID mapping for sorted pcodes
# (rasterize needs numeric values, not strings)
pcodes = sorted(gdf[pcode_col].tolist())
pcode_to_id = {p: i + 1 for i, p in enumerate(pcodes)}
id_to_pcode = {v: k for k, v in pcode_to_id.items()}

# Rasterize admin polygons onto the flood grid
# all_touched=True: include edge pixels (consistent with GFM extraction QC)
admin_shapes = [
    (geom, pcode_to_id[pcode])
    for geom, pcode in zip(gdf_proj.geometry, gdf_proj[pcode_col])
]

admin_raster = rasterize(
    admin_shapes,
    out_shape=ref_shape,
    transform=ref_transform,
    fill=0,
    dtype=np.int32,
    all_touched=True,
)

n_inside = int((admin_raster > 0).sum())
log.info(f"Admin raster: {n_inside:,} pixels inside admin boundaries")


##### 7. COMPUTE ZONAL INDICATOR COMPONENTS

In [ ]:
# Build flat arrays for zonal aggregation
# Only consider pixels inside admin boundaries
inside_mask = admin_raster > 0

admin_ids_flat = admin_raster[inside_mask]
flood_flat = cumulative_flood_days[inside_mask]
pop_flat = pop_zero_filled[inside_mask]
pop_has_data_flat = pop_with_data[inside_mask]
flooded_flat = any_flooded_mask[inside_mask].astype(bool)

# Component quantities (per pixel, flattened)
# Numerator (shared by indicators 2 & 3): flood_days × pop
hp_flat = flood_flat * pop_flat

# Population where flooded (for indicator 1 numerator & indicator 3 denominator)
pop_where_flooded_flat = np.where(flooded_flat, pop_flat, 0.0)

# Build DataFrame for groupby aggregation
pixel_df = pd.DataFrame({
    "admin_id": admin_ids_flat,
    "pop": pop_flat,                        # p_total (for indicator 2 denominator)
    "pop_where_flooded": pop_where_flooded_flat,  # p_flooded
    "hp": hp_flat,                           # numerator
    "is_flooded": flooded_flat.astype(int),  # for pixel counts
    "pop_has_data": pop_has_data_flat.astype(int),  # for QC
    "pop_data_and_flooded": (pop_has_data_flat & flooded_flat).astype(int),
})

# NOTE: is this always same as n_inside from prev. cell?
log.info(f"Pixel DataFrame: {len(pixel_df):,} rows")


In [ ]:
# Aggregate by admin area
zonal = pixel_df.groupby("admin_id").agg(
    p_total=("pop", "sum"),
    p_flooded=("pop_where_flooded", "sum"),
    hp=("hp", "sum"),
    n_pixels=("pop", "count"),
    n_flooded_pixels=("is_flooded", "sum"),
    n_pop_data_pixels=("pop_has_data", "sum"),
    n_pop_data_and_flooded=("pop_data_and_flooded", "sum"),
).reset_index()

# Map integer IDs back to pcodes
zonal[pcode_col] = zonal["admin_id"].map(id_to_pcode)

log.info(f"Zonal aggregation: {len(zonal)} admin areas")


In [ ]:
# Compute the three indicators

# Indicator 1: population rate impacted by flooding (%)
# % of admin population living under any-day-flooded mask
zonal["flood_impact_pop_pct"] = (
    zonal["p_flooded"] / zonal["p_total"] * 100
).round(2)

# Indicator 2: Pop-weighted avg flood days — total-population denominator
# "Average flood-day burden per person in this admin area"
zonal["wavg_flood_days_total_pop"] = (
    zonal["hp"] / zonal["p_total"]
).round(4)

# Indicator 3: Pop-weighted avg flood days — intersection denominator
# "Among people who experienced any flooding, average flood days"
zonal["wavg_flood_days_intersection"] = (
    zonal["hp"] / zonal["p_flooded"]
).round(4)

# QC kpi1: percentage of area flooded
zonal["pct_area_flooded"] = (
    zonal["n_flooded_pixels"] / zonal["n_pixels"] * 100
).round(2)

# QC kpi2: of flooded pixels, how many have population data behind
zonal["pct_flooded_with_pop"] = np.where(
    zonal["n_flooded_pixels"] > 0,
    (zonal["n_pop_data_and_flooded"] / zonal["n_flooded_pixels"] * 100).round(2),
    np.nan,
)

# Handle division edge cases:
# - p_total = 0 → no population → indicators = 0 (not NaN)
# - p_flooded = 0 → no one flooded → indicator 3 = NaN (but from 0/0)
# Indicator 1, 2 & 3: replace NaN from div by zero or 0/0 with 0 (no people = no impact)
zonal["flood_impact_pop_pct"] = zonal["flood_impact_pop_pct"].fillna(0)
zonal["wavg_flood_days_total_pop"] = zonal["wavg_flood_days_total_pop"].fillna(0)
zonal["wavg_flood_days_intersection"] = zonal["wavg_flood_days_intersection"].fillna(0)

# QC kpi2: leave NaN where no flooded pixels (not applicable)


In [ ]:
# Merge with full admin list (right join to keep admins with zero flood)
result_df = gdf[[pcode_col, name_col]].merge(
    zonal[
        [
            pcode_col,
            "p_total",
            "p_flooded",
            "flood_impact_pop_pct",
            "wavg_flood_days_total_pop",
            "wavg_flood_days_intersection",
            "n_pixels",
            "n_flooded_pixels",
            "pct_area_flooded",
            "n_pop_data_pixels",
            "n_pop_data_and_flooded",
            "pct_flooded_with_pop",
        ]
    ],
    on=pcode_col,
    how="left",
).sort_values(pcode_col)

# Fill admins outside flood grid coverage (if any) with zeros
for col in [
    "flood_impact_pop_pct",
    "wavg_flood_days_total_pop",
    "wavg_flood_days_intersection",
    "p_total",
    "p_flooded",
]:
    result_df[col] = result_df[col].fillna(0)

log.info(f"Result table: {len(result_df)} admin areas")
result_df.head()


##### 8. BINARIZATION

In [ ]:
def compute_binarization(series, comparison="gt"):
    """
    Binarize an indicator series: identify most-deprived admin areas.

    NOTE: For flood indicators, comparison is always "gt" (more flood = worse).
    NaN values propagate (no classification for no-data admins).
    NOTE: For flood indicators, no-data should not exist, always treated as zero.

    Returns a dict with mean, median, and the two binary Series.

    NOTE: This function is designed to be liftable into a shared wia_utils
    module for reuse across hazard post-processing workflows.
    """
    stat_mean = series.mean()
    stat_median = series.median()
    n_total = len(series)
    n_nodata = int(series.isna().sum())

    if comparison == "gt":
        prefix = "above"
        bin_mean = (series >= stat_mean).astype("Int64")
        bin_median = (series >= stat_median).astype("Int64")
    else:
        prefix = "below"
        bin_mean = (series <= stat_mean).astype("Int64")
        bin_median = (series <= stat_median).astype("Int64")

    # Ensure NaN propagation
    nodata_mask = series.isna()
    bin_mean[nodata_mask] = pd.NA
    bin_median[nodata_mask] = pd.NA

    stats = {
        "direction": prefix,
        "binarization_mean": round(stat_mean, 4) if pd.notna(stat_mean) else None,
        "binarization_median": round(stat_median, 4) if pd.notna(stat_median) else None,
        "n_regions_total": n_total,
        "n_regions_with_data": n_total - n_nodata,
        "n_regions_nodata": n_nodata,
        "n_deprived_by_mean": int(bin_mean.sum()) if not bin_mean.isna().all() else None,
        "n_deprived_by_median": int(bin_median.sum()) if not bin_median.isna().all() else None,
    }

    return f"{prefix}_mean", bin_mean, f"{prefix}_median", bin_median, stats


In [ ]:
# Binarize all three indicators
bin_stats_all = {}

indicators = [
    ("flood_impact_pop_pct", "Impacted Pop %"),
    ("wavg_flood_days_total_pop", "Wavg Flood Days (total pop)"),
    ("wavg_flood_days_intersection", "Wavg Flood Days (intersection)"),
]

for col, label in indicators:
    mean_name, bin_m, med_name, bin_md, stats = compute_binarization(
        result_df[col], comparison="gt"
    )
    result_df[f"{col}_{mean_name}"] = bin_m
    result_df[f"{col}_{med_name}"] = bin_md
    bin_stats_all[col] = stats

    log.info(
        f"  {label}: mean={stats['binarization_mean']:.4f}, "
        f"median={stats['binarization_median']:.4f}, "
        f"deprived(mean)={stats['n_deprived_by_mean']}, "
        f"nodata={stats['n_regions_nodata']}"
    )


##### 9. EXCEL OUTPUT
One for each indicator + binarization

In [ ]:
# include population year in sheet name
sheet_name = f"{c_iso3}_adm{admin_level}_{year}"

# include indicator, time window and population year in filenames
flood_files_prefix = (
    f"{c_iso3}_adm{admin_level}_{year}_"
    f"{start_query_date.strftime('%Y-%m')}_"
    f"{last_updated_month.strftime('%Y-%m')}_"
)
xlsx_filenames = [
    f"{flood_files_prefix}_{i[0]}.xlsx"
    for i in indicators
]

# Select output columns
output_cols = [
    pcode_col,
    name_col,
    # Indicator 1
    "flood_impact_pop_pct",
    "flood_impact_pop_pct_above_mean",
    "flood_impact_pop_pct_above_median",
    # Indicator 2
    "wavg_flood_days_total_pop",
    "wavg_flood_days_total_pop_above_mean",
    "wavg_flood_days_total_pop_above_median",
    # Indicator 3
    "wavg_flood_days_intersection",
    "wavg_flood_days_intersection_above_mean",
    "wavg_flood_days_intersection_above_median",
]

# force round again as not all appearing in Excel ?
# NOTE: still some numbers not exported rounded to Excel for some reason
round_indicators = [2, 4, 4]

for i in range(len(indicators)):
    xlsx_path = maps_hazards_folder + xlsx_filenames[i]

    result_df[
        output_cols[:2] + output_cols[2 + 3 * i:2 + 3 * (i + 1)]
    ].fillna("no-data").round(
        {indicators[i][0]: round_indicators[i]}
    ).to_excel(
        xlsx_path,
        index=False,
        sheet_name=sheet_name,
    )

    log.info(f"Excel saved: {xlsx_path}")


##### 10a. CHOROPLETH MAPS

In [ ]:
def save_map_png(
    df,
    geojson_dict,
    pcode_col,
    indicator_col,
    title_text,
    out_path,
    color_scale="YlOrRd",
):
    """
    Render a choropleth and write to PNG.

    Regions with NaN in the indicator are rendered as a separate grey layer
    underneath the data choropleth, making nodata regions visible.

    NOTE: This function mirrors save_map_png in postprocess_exposure_wavg.ipynb
    and is designed to be liftable into a shared wia_utils module.
    """
    has_nodata = df[indicator_col].isna().any()
    data_df = df.dropna(subset=[indicator_col])

    fig = go.Figure()

    # Trace 1: nodata regions in grey
    if has_nodata:
        nodata_df = df[df[indicator_col].isna()]
        fig.add_trace(go.Choropleth(
            geojson=geojson_dict,
            featureidkey=f"properties.{pcode_col}",
            locations=nodata_df[pcode_col],
            z=[0] * len(nodata_df),
            colorscale=[[0, NODATA_GREY], [1, NODATA_GREY]],
            showscale=False,
            hoverinfo="location+text",
            text=["No data"] * len(nodata_df),
            marker_line_color="Gainsboro",
            marker_line_width=0.5,
        ))

    # Trace 2: data regions with colour scale
    fig.add_trace(go.Choropleth(
        geojson=geojson_dict,
        featureidkey=f"properties.{pcode_col}",
        locations=data_df[pcode_col],
        z=data_df[indicator_col],
        colorscale=color_scale,
        colorbar=dict(len=0.65),
        marker_line_color="Gainsboro",
        marker_line_width=0.5,
    ))

    fig.update_geos(
        fitbounds="locations",
        visible=False,
        projection_type="mercator",
    )
    fig.update_layout(
        title=dict(text=title_text, x=0.5, y=0.97),
        margin={"r": 0, "t": 0, "l": 0, "b": 0},
        width=1200,
        height=900,
    )

    fig.write_image(out_path, format="png", scale=2)


In [ ]:
# Disable automatic rendering
pio.renderers.default = None

suffix_label = flood_files_prefix.removesuffix("_")

map_configs = [
    {
        "col": "flood_impact_pop_pct",
        "title": f"Impacted Pop [%] — GFM Flood — {suffix_label}",
        "file_suffix": indicators[0][0],
    },
    {
        "col": "wavg_flood_days_total_pop",
        "title": f"Pop-weighted Avg Flood Days (all pop) — GFM — {suffix_label}",
        "file_suffix": indicators[1][0],
    },
    {
        "col": "wavg_flood_days_intersection",
        "title": f"Pop-weighted Avg Flood Days (flooded pop) — GFM — {suffix_label}",
        "file_suffix": indicators[2][0],
    },
]

for cfg in map_configs:
    png_path = maps_hazards_folder + f"{suffix_label}_{cfg['file_suffix']}.png"
    save_map_png(
        result_df,
        geojson_dict,
        pcode_col,
        cfg["col"],
        cfg["title"],
        png_path,
        color_scale=CSCALE_FLOOD,
    )
    log.info(f"  Map saved: {png_path}")


##### 10b. Flood Mask Overlay Map
NOTE: raster-based, should be adapted if vector-based (geojson) used eventually

In [ ]:
# 1. Reproject the binary mask to EPSG:4326
#    Compute the destination grid dimensions from the country bounds
#    at a resolution coarse enough for visualization (~0.5/1km equivalent)
#    This downsamples implicitly, keeping point count manageable.

# Country bounds in 4326 (from gdf)
bounds_4326 = gdf.total_bounds  # minx, miny, maxx, maxy

dst_width = int((bounds_4326[2] - bounds_4326[0]) / VIZ_RES_DEG)
dst_height = int((bounds_4326[3] - bounds_4326[1]) / VIZ_RES_DEG)

dst_transform = from_bounds(
    *bounds_4326, dst_width, dst_height
)

mask_4326 = np.zeros((dst_height, dst_width), dtype=np.uint8)

reproject(
    source=any_flooded_mask,
    destination=mask_4326,
    src_transform=ref_transform,
    src_crs=ref_crs,
    dst_transform=dst_transform,
    dst_crs="EPSG:4326",
    resampling=Resampling.max,  # any flooded source pixel promotes the coarser cell
)

# 2. Extract lon/lat of flooded pixels
rows, cols = np.where(mask_4326 > 0)
lons, lats = xy(dst_transform, rows, cols)

log.info(
    f"Flood mask for viz: {len(lons):,} points "
    f"(from {dst_height}x{dst_width} grid at ~{VIZ_RES_DEG*111000:.0f}m)"
)


In [ ]:
# ── Flood mask overlay map ──
fig = go.Figure()

# exclude population year in flood area map
flood_area_label = (
    f"{c_iso3}_adm{admin_level}_"
    f"{start_query_date.strftime('%Y-%m')}_"
    f"{last_updated_month.strftime('%Y-%m')}"
)

# ── Layer 1: Admin boundaries on a light background ──
# Uniform white fill so the flood overlay pops visually
fig.add_trace(go.Choropleth(
    geojson=geojson_dict,
    featureidkey=f"properties.{pcode_col}",
    locations=gdf[pcode_col],
    z=[0] * len(gdf),
    colorscale=[[0, "white"], [1, "white"]],
    showscale=False,
    marker_line_color="DarkGrey",
    marker_line_width=1,
    hoverinfo="skip",
))

# ── Layer 2: Flood mask as scatter points ──
# Each point is one ~0.5/1km pixel; at country zoom they merge into
# a continuous flood footprint.
fig.add_trace(go.Scattergeo(
    lon=list(lons),
    lat=list(lats),
    mode="markers",
    marker=dict(
        size=1,
        color="dodgerblue",
        opacity=0.35,
    ),
    name="Flooded area",
    hoverinfo="name",
))

fig.update_geos(
    fitbounds="locations",
    visible=False,
    projection_type="mercator",
    bgcolor="white",
)

fig.update_layout(
    title=dict(
        text=f"Any-day Flooded Area — GFM {flood_area_label}",
        x=0.5,
        y=0.97,
    ),
    margin={"r": 0, "t": 30, "l": 0, "b": 0},
    width=1200,
    height=900,
    paper_bgcolor="white",
    showlegend=False,
    # legend=dict(
    #     x=0.02, y=0.02,
    #     bgcolor="rgba(255,255,255,0.8)",
    # ),
)

# ── Save ──
png_path = maps_hazards_folder + f"{flood_area_label}_GFM_flood_mask_viz.png"
fig.write_image(png_path, format="png", scale=2)
log.info(f"Flood mask map saved: {png_path}")


##### 11. JSON RUN LOG

In [ ]:
# Build per-admin QC records
admin_qc = []
for _, row in result_df.iterrows():
    entry = {
        "pcode": row[pcode_col],
        "name": row[name_col],
        "total_pixels": int(row["n_pixels"]) if pd.notna(row.get("n_pixels")) else 0,
        "flooded_pixels": int(row["n_flooded_pixels"]) if pd.notna(row.get("n_flooded_pixels")) else 0,
        "pct_area_flooded": float(row["pct_area_flooded"]) if pd.notna(row.get("pct_area_flooded")) else 0,
        "pop_data_pixels": int(row["n_pop_data_pixels"]) if pd.notna(row.get("n_pop_data_pixels")) else 0,
        "flooded_pixels_with_pop_data": (
            int(row["n_pop_data_and_flooded"]) if pd.notna(row.get("n_pop_data_and_flooded")) else 0
        ),
        "pct_flooded_with_pop_data": (
            float(row["pct_flooded_with_pop"])
            if pd.notna(row.get("pct_flooded_with_pop")) else None
        ),
        "total_population": round(float(row["p_total"]), 1),
        "impacted_population": round(float(row["p_flooded"]), 1),
        "impacted_pop_pct": float(row["flood_impact_pop_pct"]),
        "wavg_flood_days_total_pop": (
            float(row["wavg_flood_days_total_pop"])
            if pd.notna(row["wavg_flood_days_total_pop"]) else 0
        ),
        "wavg_flood_days_intersection": (
            float(row["wavg_flood_days_intersection"])
            if pd.notna(row["wavg_flood_days_intersection"]) else 0
        ),
    }
    admin_qc.append(entry)


In [ ]:
# Assemble the run log
run_log = {
    "pipeline": "postprocess_gfm_flood",
    "run_timestamp": datetime.now(timezone.utc).isoformat(),

    # ── Parameters ──
    "country": country,
    "iso3": c_iso3,
    "admin_level": admin_level,
    "population_year": year,
    "shapefile": geojson_file[0],
    "n_admin_regions": len(gdf),
    "population_raster": pop_raster_name,
    "flood_raster_dir": str(gfm_raster_dir),
    "flood_crs": str(ref_crs),
    "flood_grid_shape": list(ref_shape),
    "months_stacked": months_loaded,
    "n_months": len(months_loaded),

    # ── Processing choices ──
    "pop_resampling": "nearest",
    "admin_rasterize_all_touched": True,
    "pop_nodata_treatment": "zero_filled (constrained pop: nodata = no population)",

    # ── Global flood mask stats ──
    "flood_mask": {
        "n_flooded_pixels": n_flooded_pixels,
        # "geojson_uncompressed_mb": round(uncompressed_mb, 2),
        # "geojson_visualization_mb": round(compressed_mb, 2),
        # "geojson_visualization_path": str(geo_path),
    },

    # ── Population summary ──
    "total_pop_in_country": round(float(np.nansum(pop_aligned)), 1),
    "total_pop_in_admin_areas": round(float(result_df["p_total"].sum()), 1),
    "total_pop_exposed_to_flood": round(float(result_df["p_flooded"].sum()), 1),

    # ── Binarization statistics ──
    "binarization": bin_stats_all,

    # ── Per-admin QC ──
    "admin_qc": admin_qc,

    # ── Output files ──
    "outputs": {
        "xlsx": xlsx_path,
        "maps": [
            maps_hazards_folder + f"{suffix_label}_{cfg['file_suffix']}.png"
            for cfg in map_configs
        ],
        # "flood_mask_geojson": str(geo_path),
    },
}

# Write log
log_path = output_log_dir + f"postprocess_gfm_flood_{suffix_label}.json"
with open(log_path, "w") as f:
    json.dump(run_log, f, indent=4, default=str)

log.info(f"Run log saved: {log_path}")
log.info("Post-processing complete.")
